## Build Daily Aggregates Table
Load the cleaned transactions, aggregate by day to compute revenue, average price, transaction counts, unique customers/articles, and online purchase percentage. Fill missing dates with zeros and save the daily aggregates file.

In [ ]:
import numpy as np
import pandas as pd

# Load cleaned data
df = pd.read_csv('hm_clean.csv', dtype={'article_id':str, 'customer_id':str}, parse_dates=['t_dat'])
print(df.head(5).to_string())

       t_dat                                                       customer_id  article_id     price  sales_channel_id club_member_status fashion_news_frequency   age                                                       postal_code  product_code               prod_name product_type_name  product_group_name graphical_appearance_name colour_group_name perceived_colour_value_name perceived_colour_master_name          department_name              index_name index_group_name                    section_name garment_group_name                                                                                                                                                                                                                                   detail_desc  is_fn_subscriber  is_active_customer  has_description  price_log
0 2019-09-13  215895f90002eb3d1a04bd603513c8e85e6002ef08f13651b65631e0db7d1644  0786586001  0.022017                 1             active                   none  41.0  5

In [10]:
# Group by date (not datetime) and compute daily metrics
df['date'] = df['t_dat'].dt.date
daily = df.groupby('date').agg(
    revenue=('price','sum'),
    avg_price=('price','mean'),
    num_transactions=('price','count'),
    num_customers=('customer_id','nunique'),
    num_articles=('article_id','nunique'),
    online_pct=('sales_channel_id', lambda x: (x==2).sum() / len(x) * 100)
).reset_index()

# Convert date to datetime and fill missing dates with zeros
daily['date'] = pd.to_datetime(daily['date'])
daily = daily.set_index('date').asfreq('D').fillna(0).reset_index()

# Flatten column order and rename if needed
daily.columns = ['date','revenue','avg_price','num_transactions','num_customers','num_articles','online_pct']

# Basic checks and save
print(f"Daily aggregates shape: {daily.shape}")
print(f"Date range: {daily['date'].min().date()} to {daily['date'].max().date()}")
print(f"Days with transactions: {(daily['num_transactions']>0).sum()} / {len(daily)}")
print('\nFirst 5 rows:')
print(daily.head())

daily.to_csv('daily_aggregates.csv', index=False)
print('\nSaved daily_aggregates.csv')

Daily aggregates shape: (734, 7)
Date range: 2018-09-20 to 2020-09-22
Days with transactions: 734 / 734

First 5 rows:
        date   revenue  avg_price  num_transactions  num_customers  \
0 2018-09-20  0.346780   0.028898                12             12   
1 2018-09-21  0.271492   0.038785                 7              7   
2 2018-09-22  0.084695   0.028232                 3              3   
3 2018-09-23  0.268458   0.024405                11             11   
4 2018-09-24  0.712153   0.039564                18             18   

   num_articles  online_pct  
0            12   91.666667  
1             7   71.428571  
2             3   33.333333  
3            11   81.818182  
4            18   83.333333  

Saved daily_aggregates.csv


## Add Temporal Features
Load the daily aggregates and add calendar features (day, month, quarter, week), seasonality flags, rolling averages, lag features, and percent change to support time-series modeling.

In [ ]:
# Load daily aggregates and numpy
daily = pd.read_csv('daily_aggregates.csv', parse_dates=['date'])

# Calendar features: day, month, quarter, week, year, weekend flag
daily['day_of_week'] = daily['date'].dt.dayofweek
daily['month'] = daily['date'].dt.month
daily['quarter'] = daily['date'].dt.quarter
daily['week_of_year'] = daily['date'].dt.isocalendar().week.astype(int)
daily['is_weekend'] = (daily['day_of_week'] >= 5).astype(int)
daily['year'] = daily['date'].dt.year

# Cyclic encodings for day_of_week and month
daily['dow_sin'] = np.sin(2 * np.pi * daily['day_of_week'] / 7)
daily['dow_cos'] = np.cos(2 * np.pi * daily['day_of_week'] / 7)
daily['month_sin'] = np.sin(2 * np.pi * (daily['month'] - 1) / 12)
daily['month_cos'] = np.cos(2 * np.pi * (daily['month'] - 1) / 12)

# Season / holiday flags
daily['is_summer'] = daily['month'].isin([6,7,8]).astype(int)
daily['is_winter'] = daily['month'].isin([12,1,2]).astype(int)
daily['is_holiday_season'] = daily['month'].isin([11,12]).astype(int)

# Rolling statistics (mean, std, min, max) for windows 7,14,30
for w in (7,14,30):
    daily[f'rev_roll_mean_{w}'] = daily['revenue'].rolling(window=w, min_periods=1).mean()
    daily[f'rev_roll_std_{w}'] = daily['revenue'].rolling(window=w, min_periods=1).std().fillna(0)
    daily[f'rev_roll_min_{w}'] = daily['revenue'].rolling(window=w, min_periods=1).min()
    daily[f'rev_roll_max_{w}'] = daily['revenue'].rolling(window=w, min_periods=1).max()

# EWMA
daily['rev_ewm_span14'] = daily['revenue'].ewm(span=14, adjust=False).mean()

# Lag features and lag-aggregates
daily['rev_lag1'] = daily['revenue'].shift(1)
daily['rev_lag7'] = daily['revenue'].shift(7)
daily['rev_lag14'] = daily['revenue'].shift(14)
daily['rev_lag30'] = daily['revenue'].shift(30)
daily['lag7_mean'] = daily['revenue'].shift(1).rolling(window=7, min_periods=1).mean()
daily['lag14_mean'] = daily['revenue'].shift(1).rolling(window=14, min_periods=1).mean()
daily['lag7_std'] = daily['revenue'].shift(1).rolling(window=7, min_periods=1).std().fillna(0)

# Percent / absolute change features
daily['rev_pct_change_1d'] = daily['revenue'].pct_change(periods=1).replace([np.inf, -np.inf], np.nan) * 100
daily['rev_pct_change_7d'] = daily['revenue'].pct_change(periods=7).replace([np.inf, -np.inf], np.nan) * 100

# MTD and WTD cumulative revenue
daily = daily.sort_values('date').reset_index(drop=True)
daily['mtd_revenue'] = daily.groupby(['year','month'])['revenue'].cumsum()
daily['wtd_revenue'] = daily.groupby(['year', daily['date'].dt.isocalendar().week])['revenue'].cumsum()

# Outlier flags using rolling z-score (30-day window)
roll_mean_30 = daily['revenue'].rolling(window=30, min_periods=7).mean()
roll_std_30 = daily['revenue'].rolling(window=30, min_periods=7).std().replace(0, np.nan)
daily['rev_rolling_zscore_30'] = (daily['revenue'] - roll_mean_30) / roll_std_30
daily['is_rev_outlier_30'] = daily['rev_rolling_zscore_30'].abs() > 3
daily['is_rev_outlier_30'] = daily['is_rev_outlier_30'].fillna(False)

# STL decomposition if available (graceful fallback)
try:
    from statsmodels.tsa.seasonal import STL
    stl = STL(daily['revenue'].fillna(0), period=7, robust=True)
    res = stl.fit()
    daily['rev_stl_trend'] = res.trend
    daily['rev_stl_seasonal'] = res.seasonal
    daily['rev_stl_resid'] = res.resid
except Exception:
    daily['rev_stl_trend'] = np.nan
    daily['rev_stl_seasonal'] = np.nan
    daily['rev_stl_resid'] = np.nan

# Top-N department daily revenue pivots merged back 
try:
    df = pd.read_csv('hm_clean.csv', parse_dates=['t_dat'], dtype={'article_id':str,'customer_id':str})
    df['date'] = df['t_dat'].dt.date
    top_depts = (df.groupby('department_name')['price'].sum().sort_values(ascending=False).head(3).index.tolist())
    dept_daily = df[df['department_name'].isin(top_depts)].groupby(['date','department_name'])['price'].sum().unstack(fill_value=0)
    dept_daily.index = pd.to_datetime(dept_daily.index)
    dept_daily = dept_daily.reindex(daily['date']).fillna(0).reset_index(drop=True)
    for col in dept_daily.columns:
        daily[f'dept_rev_{col}'] = dept_daily[col].values
except Exception:
    pass

# Imputation / status flags
daily['no_sales_day'] = (daily['num_transactions'] == 0).astype(int)
daily['rev_was_zero'] = (daily['revenue'] == 0).astype(int)

# Finalize and save
daily = daily.sort_values('date').reset_index(drop=True)
daily.to_csv('daily_aggregates.csv', index=False)
print(f"Enhanced features added. Shape: {daily.shape}")

Enhanced features added. Shape: (734, 54)
